In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ["OPENAI_API_KEY"]:
    print("API Key is set.")

API Key is set


In [3]:
from langchain_openai import ChatOpenAI

In [4]:
llm = ChatOpenAI(model="gpt-5-nano",temperature=0)

In [5]:
response = llm.invoke("What is AI? tell me in one line and short.")

In [7]:
response

AIMessage(content='AI is the science of building machines that can perform tasks that normally require human intelligence.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 346, 'prompt_tokens': 18, 'total_tokens': 364, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EX0jbG1gzqBrhehrr07ejomPKYgcA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a11fe3-5072-79e3-9166-6831dcfac9c1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 346, 'total_tokens': 364, 'input_token_details': {'audio': 0, 'cache_read': 0

# RAG

### Step 1: Preparing Doc for Text

In [9]:
from langchain_core.documents import Document

my_text = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]

Artificial intelligence was founded as an academic discipline in 1956.[4] The field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Funding and interest increased substantially after 2012, when graphics processing units (GPUs) started being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[9] This growth accelerated further after 2017 with the transformer architecture.[10] In the 2020s, an AI boom coincided with advances in generative AI, which became widespread and allowed for the creation and modification of media. In addition to AI safety and unintended consequences and harms from the use of AI, ethical concerns, AI's long-term effects, environmental effects,[11] and potential existential risks have prompted discussions of AI regulation."""

In [13]:
docs = [Document(page_content=my_text, metadata={"source":"ABC", "documentID":"Doc1"})]

docs

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content="Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).\n\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a

### Step 2: Splitting the Document into CHUNKS

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='The traditional goals of AI resea

### Step 3: Creating Embeddings for the Chunks

In [21]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

### Step 4: Create and Store Embeddings in Vector Store

In [27]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

### Step 5: Semantic Search

In [28]:
vectorstore.similarity_search("What is AI?", k=3)

[Document(metadata={'documentID': 'Doc1', 'source': 'ABC'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='The traditional goals of AI resea

### Talk to LLM

In [29]:
context = vectorstore.similarity_search("What is AI?", k=3)

In [30]:
response = llm.invoke(f"What is AI? You can answer using the following context {context}")

In [32]:
response.content

'Artificial intelligence (AI) is the capability of computer systems to perform tasks that typically require human intelligence. This includes learning, reasoning, problem-solving, perception, and decision-making. AI is a field of engineering, mathematics, and computer science that develops methods and software enabling machines to perceive their environment, learn, and take actions to achieve defined goals.\n\nKey points:\n- Goals of AI: learning, reasoning, knowledge representation, planning, natural language processing, perception, and robotics support.\n- Common techniques: state-space search, mathematical optimization, formal logic, artificial neural networks, statistics, operations research, and economics.\n- Applications: web search engines, chatbots and virtual assistants, autonomous vehicles, strategy game play (e.g., chess, Go), and content generation (text, images, audio, video).'